In [ ]:
# Librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso, RidgeCV, LassoCV
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
# Cargamos el dataset
drive.mount('/content/drive')

df = pd.read_csv("/content/drive/MyDrive/ML/company.csv")  # Modificar según ruta donde se almacene

print(df.head()) # Los datos cargaron correctamente


In [ ]:
# Realizamos un breve ánalisis exploratorio
df.info()

In [ ]:
df.describe()

In [ ]:
df.boxplot()

El dataset presenta cuatro variables: TV, Radio, Newspaper y Sales. No contiene valores nulos y todas las variables son floats. TV muestra una mayor variabilidad y escala en comparación con las otras. En general, las variables no parecen presentar una cantidad significativa de outliers.

In [ ]:
corr_matrix = df.corr()
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', center=0)

Se muestran diversas correlaciones de intensidad moderada-baja. Ventas y TV presentan una correlación positiva.

In [ ]:
X = df.drop("Sales", axis=1)
y = df["Sales"]

# Dividimos el data set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=33)

# Normalizamos
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)

In [ ]:
# Ridge
alphas = np.logspace(-10, 2, 200)


ridge_cv = RidgeCV(alphas=alphas, cv=5)
ridge_cv.fit(X_train, y_train)

y_pred_ridge = ridge_cv.predict(X_test)
mse_ridge_test = mean_squared_error(y_test, y_pred_ridge)
r2_ridge_test = r2_score(y_test, y_pred_ridge)


In [ ]:
# Lasso
lasso_cv = LassoCV(alphas=alphas, cv=5, max_iter=10000)
lasso_cv.fit(X_train, y_train)

y_pred_lasso = lasso_cv.predict(X_test)
mse_lasso_test = mean_squared_error(y_test, y_pred_lasso)
r2_lasso_test = r2_score(y_test, y_pred_lasso)

In [ ]:
# Comparamos los resultados
print("--- RESULTADOS RIDGE ---")
print(f"Mejor alpha: {ridge_cv.alpha_}")
print(f"MSE test: {mse_ridge_test:.4f} | R² test: {r2_ridge_test:.4f}")
print(f"Coef: {ridge_cv.coef_}")

print("\n--- RESULTADOS LASSO ---")
print(f"Mejor alpha: {lasso_cv.alpha_}")
print(f"MSE test: {mse_lasso_test:.4f} | R² test: {r2_lasso_test:.4f}")
print(f"Coef: {lasso_cv.coef_}")

En el caso de Lasso, como era de esperar, el coeficiente de la variable Newspaper es 0, mientras que en Ridge también se mantiene muy bajo. Ambos modelos muestran un rendimiento muy similar, aunque el mejor alpha de Lasso es más pequeño (0.02) frente a Ridge (1.7). Ambos alcanzan un nivel de significación alrededor de 0.91.

A medida que aumentamos α, los coeficientes tienden a acercarse a 0, en el caso de Lasso, algunas variables incluso llegan a desaparecer por completo.

Ridge es más “flexible”, ya que no tiende a eliminar variables, mientras que Lasso sí puede dejar fuera aquellas que no aportan mucho al modelo.

En cuanto al rendimiento, ambos son muy similares, pero siguiendo la ley de parsimonia, Lasso puede ser preferible porque genera modelos más simples.

La regularización ayuda a que las variables menos importantes tengan menos peso. Además, en situaciones de multicolinealidad o riesgo de sobreajuste, mantiene los coeficientes más estables, evitando que se disparen demasiado con los datos de entrenamiento o con una fuerte correlación.

En datasets con muchas variables o con varias irrelevantes, Lasso suele ser preferible porque tiende a eliminar las que no aportan al modelo. En cambio, si el conjunto de datos es pequeño o se desea conservar todas las variables, Ridge resulta más conveniente.

In [ ]:
alphas = ridge_cv.alphas
coefs = []

for alpha in alphas:
    modelo_temp = Ridge(alpha=alpha, fit_intercept=False)
    modelo_temp.fit(X_train, y_train)
    coefs.append(modelo_temp.coef_.flatten())

fig, ax = plt.subplots(figsize=(7, 3.84))
ax.plot(alphas, coefs)
ax.set_xscale('log')
ax.set_xlabel('alpha')
ax.set_ylabel('coeficientes')
ax.set_title('Coeficientes del modelo en función de la regularización Ridge');
plt.axis('tight')
plt.show()

In [ ]:
alphas = lasso_cv.alphas
coefs = []

for alpha in alphas:
    modelo_temp = Lasso(alpha=alpha, fit_intercept=False)
    modelo_temp.fit(X_train, y_train)
    coefs.append(modelo_temp.coef_.flatten())

fig, ax = plt.subplots(figsize=(7, 3.84))
ax.plot(alphas, coefs)
ax.set_xscale('log')
ax.set_xlabel('alpha')
ax.set_ylabel('coeficientes')
ax.set_title('Coeficientes del modelo en función de la regularización Lasso');
plt.axis('tight')
plt.show()

Se observa que el valor de los coeficientes no varía demasiado en ambos modelos y a partir de qué valores Lasso drásticamente los pone a 0.